In [1]:
import sys
import os
sys.path.append(os.path.abspath('../../src'))

In [2]:
import os
from torch.utils.data import Dataset
from torch.utils.data.dataloader import DataLoader
import torch.nn.functional as F
from losses import edl_mse_loss

from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
from tqdm.auto import tqdm

from dataset import TextDataset 
from losses import edl_mse_loss

import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix
from sklearn.model_selection import train_test_split
from torch.utils.data import Subset

from probes import ProbeClassification, ProbeClassificationMixScaler
from train_test_utils import train, test 
import torch.nn as nn

import time

tic, toc = (time.time, time.time)

In [3]:
# tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-2-13b-chat-hf", use_auth_token=True)
# model = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-2-13b-chat-hf", use_auth_token=True)
# model.half().cuda();
# model.eval();
tokenizer = AutoTokenizer.from_pretrained("openai/gpt-oss-20b")

model = AutoModelForCausalLM.from_pretrained(
    "openai/gpt-oss-20b",
    torch_dtype=torch.float16,
    device_map="auto",
)
model.eval()

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/27.9M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/98.0 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

model-00000-of-00002.safetensors:   0%|          | 0.00/4.79G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.80G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/4.17G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 1.98 GiB. GPU 0 has a total capacity of 19.62 GiB of which 350.88 MiB is free. Including non-PyTorch memory, this process has 19.17 GiB memory in use. Of the allocated memory 15.83 GiB is allocated by PyTorch, and 3.14 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
from probes import ProbeClassification, ProbeClassificationMixScaler
    
class TrainerConfig:
    # optimization parameters
    learning_rate = 1e-3
    betas = (0.9, 0.95)
    weight_decay = 0.1 # only applied on matmul weights
    # learning rate decay params: linear warmup followed by cosine decay to 10% of original
    # checkpoint settings

    def __init__(self, **kwargs):
        for k,v in kwargs.items():
            setattr(self, k, v)

## Linear Probing on All Attributes

In [ ]:
import os
from torch.utils.data import Dataset
from torch.utils.data.dataloader import DataLoader
import torch.nn.functional as F

import torch
from tqdm.auto import tqdm
from dataset import split_conversation, llama_v2_prompt, TextDataset

### Reading Probes

In [ ]:
from probes import LinearProbeClassification, LinearProbeClassificationMixScaler
import sklearn.model_selection
import pickle
import random

jump_socioeco = True

new_prompt_format=True
residual_stream=True
uncertainty = False
logistic = True
augmented = False
remove_last_ai_response = True
include_inst = True
one_hot = True

label_to_id_age = {"child": 0,
                   "adolescent": 1,
                   "adult": 2,
                   "older adult": 3,
                  }

label_to_id_gender = {"male": 0,
                      "female": 1,
                     }

label_to_id_socioeconomic = {"low": 0,
                             "middle": 1,
                             "high": 2}

label_to_id_neweducation = {"someschool": 0,
                            "highschool": 1,
                            "collegemore": 2}
label_to_id_priority = {
    "platform": 0,
    "developer": 1,
    "user": 2
}

prompt_translator = {"_age_": "age",
                     "_gender_": "gender",
                     "_socioeco_": "socioeconomic status",
                     "_education_": "education level",
                     "_priority_": "priority level",
                    }

openai_dataset = {"_age_": "../../data/dataset/openai_age_1/",
                  "_gender_": "../../data/dataset/openai_gender_1/",
                  "_education_": "../../data/dataset/openai_education_1/",
                  "_socioeco_": "../../data/dataset/openai_socioeconomic_1/",
                  "_priority_": "../../data/dataset/openai_priority_academic_dishonesty",
                 }


accuracy_dict = {}

# directories = ["../../data/dataset/llama_age_1/", "../../data/dataset/llama_gender_1/",
#                "../../data/dataset/llama_socioeconomic_1/", "../../data/dataset/openai_education_1/"
#               ]
directories = ["../../data/dataset/openai_priority_academic_dishonesty/"]

# label_idfs = ["_age_", "_gender_", "_socioeco_", "_education_", "_priority_"]
label_idfs = ["_priority_"]

# label_to_ids = [label_to_id_age, label_to_id_gender,
#                 label_to_id_socioeconomic, label_to_id_neweducation,
#                 label_to_id_priority,
#                ]
label_to_ids = [label_to_id_priority]

for directory, label_idf, label_to_id in zip(directories, label_idfs, label_to_ids):
    # additional_dataset=[directory[:-1] + "_additional/"]
    if label_idf == "_education_":
        additional_dataset=[]
    elif label_idf == "_priority_":
        additional_dataset=[]
    else:
        additional_dataset=[directory[:-2] + "2/", openai_dataset[label_idf]]
    if label_idf == "_gender_":
        additional_dataset += ["../../data/dataset/openai_gender_2/", "../../data/dataset/openai_gender_3/", 
                               "../../data/dataset/openai_gender_4",]
    if label_idf == "_education_":
        additional_dataset += ["../../data/dataset/openai_education_2", "../../data/dataset/openai_education_3/"]
    if label_idf == "_socioeco_":
        additional_dataset += ["../../data/dataset/openai_socioeconomic_2/", "../../data/dataset/openai_socioeconomic_3/"]
    if label_idf == "_age_":
        additional_dataset += ["../../data/dataset/openai_age_2/"]
    if label_idf == "_priority_":
        # additional_dataset += [
        #     "../../data/dataset/openai_priority_addiction_facilitation/",
        #     "../../data/dataset/openai_priority_animal_harm/", 
        #     "../../data/dataset/openai_priority_autonomous_agent_scope_creep/",
        #     "../../data/dataset/openai_priority_bioweapons/",
        #     "../../data/dataset/openai_priority_child_custody/",
        #     "../../data/dataset/openai_priority_child_safety/",
        #     "../../data/dataset/openai_priority_competitor_mentions/",
        #     "../../data/dataset/openai_priority_confidential_business_information/",
        #     "../../data/dataset/openai_priority_copyright/",
        #     "../../data/dataset/openai_priority_cybersecurity/",
        #     "../../data/dataset/openai_priority_disinformation_campaign/",
        #     "../../data/dataset/openai_priority_disordered_eating/",
        #     "../../data/dataset/openai_priority_diy_activities/",
        #     "../../data/dataset/openai_priority_drug_synthesis/",
        #     "../../data/dataset/openai_priority_euthanasia/",
        #     "../../data/dataset/openai_priority_extremism/",
        #     "../../data/dataset/openai_priority_finance/",
        #     "../../data/dataset/openai_priority_financial_fraud/",
        #     "../../data/dataset/openai_priority_gambling_and_addiction/",
        #     "../../data/dataset/openai_priority_hate_speech/",
        #     "../../data/dataset/openai_priority_identity/",
        #     "../../data/dataset/openai_priority_illegal_modifications/",
        #     "../../data/dataset/openai_priority_immigration/",
        #     "../../data/dataset/openai_priority_impersonation/",
        #     "../../data/dataset/openai_priority_legal/",
        #     "../../data/dataset/openai_priority_legal_obstruction/",
        #     "../../data/dataset/openai_priority_medical/",
        #     "../../data/dataset/openai_priority_mental_health_response/",
        #     "../../data/dataset/openai_priority_minor_sexual_content/",
        #     "../../data/dataset/openai_priority_misinformation/",
        #     "../../data/dataset/openai_priority_national_security/",
        #     "../../data/dataset/openai_priority_paywall_circumvention/",
        #     "../../data/dataset/openai_priority_political_opinions/",
        #     "../../data/dataset/openai_priority_privacy/",
        #     "../../data/dataset/openai_priority_profanity/",
        #     "../../data/dataset/openai_priority_psychological_manipulation/",
        #     "../../data/dataset/openai_priority_racial_stereotyping/",
        #     "../../data/dataset/openai_priority_religion/",
        #     "../../data/dataset/openai_priority_roleplay/",
        #     "../../data/dataset/openai_priority_scope_restriction/",
        #     "../../data/dataset/openai_priority_self-harm/",
        #     "../../data/dataset/openai_priority_sexual_content/",
        #     "../../data/dataset/openai_priority_surveillance/",
        #     "../../data/dataset/openai_priority_synthetic_media/",
        #     "../../data/dataset/openai_priority_system_prompt_confidentiality/",
        #     "../../data/dataset/openai_priority_underage_alcohol_drugs/",
        #     "../../data/dataset/openai_priority_user_data/",
        #     "../../data/dataset/openai_priority_weapons_manufacturing/",
        #     "../../data/dataset/openai_priority_workplace_harassment/",
        # ]
        additional_dataset += [
            "../../data/dataset/openai_priority_addiction_facilitation/",
            "../../data/dataset/openai_priority_animal_harm/", 
            "../../data/dataset/openai_priority_autonomous_agent_scope_creep/",
            "../../data/dataset/openai_priority_bioweapons/",
            "../../data/dataset/openai_priority_child_custody/",
            "../../data/dataset/openai_priority_child_safety/",
            "../../data/dataset/openai_priority_competitor_mentions/",
        ]
        
    dataset = TextDataset(directory, tokenizer, model, label_idf=label_idf, label_to_id=label_to_id,
                          convert_to_llama2_format=True, additional_datas=additional_dataset, 
                          new_format=new_prompt_format,
                          residual_stream=residual_stream, if_augmented=augmented, 
                          remove_last_ai_response=remove_last_ai_response, include_inst=include_inst, k=1,
                          one_hot=False, last_tok_pos=-1)
    dict_name = label_idf.strip("_")

    # train_size = int(0.8 * len(dataset))
    # test_size = len(dataset) - train_size
    train_idx, val_idx = sklearn.model_selection.train_test_split(list(range(len(dataset))), 
                                                                  test_size=0.2,
                                                                  train_size=0.8,
                                                                  random_state=12345,
                                                                  shuffle=True,
                                                                  stratify=dataset.labels,
                                                                 )

    train_dataset = Subset(dataset, train_idx)
    test_dataset = Subset(dataset, val_idx)

    sampler = None
    train_loader = DataLoader(train_dataset, shuffle=True, sampler=sampler, pin_memory=True, batch_size=200, num_workers=1)
    test_loader = DataLoader(test_dataset, shuffle=False, pin_memory=True, batch_size=400, num_workers=1)

    if uncertainty:
        loss_func = edl_mse_loss
    else:
        loss_func = nn.BCELoss()
    torch_device = "cuda"

    seeds = list(range(42))
    seeds = seeds[:9]
    accuracy_dict[dict_name] = []
    accuracy_dict[dict_name + "_final"] = []
    accuracy_dict[dict_name + "_train"] = []
        
    accs = []
    final_accs = []
    train_accs = []
    for i in tqdm(range(0, 41)):
        trainer_config = TrainerConfig()
        probe = LinearProbeClassification(probe_class=len(label_to_id.keys()), device="cuda", input_dim=5120,
                                            logistic=logistic)
        optimizer, scheduler = probe.configure_optimizers(trainer_config)
        best_acc = 0
        max_epoch = 50
        verbosity = False
        layer_num = i
        print("-" * 40 + f"Layer {layer_num}" + "-" * 40)
        for epoch in range(1, max_epoch + 1):
            if epoch == max_epoch:
                verbosity = True
            # Get the train results from training of each epoch
            if uncertainty:
                train_results = train(probe, torch_device, train_loader, optimizer, 
                                        epoch, loss_func=loss_func, verbose_interval=None,
                                        verbose=verbosity, layer_num=layer_num, 
                                        return_raw_outputs=True, epoch_num=epoch, num_classes=len(label_to_id.keys()))
                test_results = test(probe, torch_device, test_loader, loss_func=loss_func, 
                                    return_raw_outputs=True, verbose=verbosity, layer_num=layer_num,
                                    scheduler=scheduler, epoch_num=epoch, num_classes=len(label_to_id.keys()))
            else:
                train_results = train(probe, torch_device, train_loader, optimizer, 
                                        epoch, loss_func=loss_func, verbose_interval=None,
                                        verbose=verbosity, layer_num=layer_num,
                                        return_raw_outputs=True,
                                        one_hot=one_hot, num_classes=len(label_to_id.keys()))
                test_results = test(probe, torch_device, test_loader, loss_func=loss_func, 
                                    return_raw_outputs=True, verbose=verbosity, layer_num=layer_num,
                                    scheduler=scheduler,
                                    one_hot=one_hot, num_classes=len(label_to_id.keys()))

            if test_results[1] > best_acc:
                best_acc = test_results[1]
                torch.save(probe.state_dict(), f"../../data/probe_checkpoints/reading_probe/{dict_name}_probe_at_layer_{layer_num}.pth")
        torch.save(probe.state_dict(), f"../../data/probe_checkpoints/reading_probe/{dict_name}_probe_at_layer_{layer_num}_final.pth")
        
        accs.append(best_acc)
        final_accs.append(test_results[1])
        train_accs.append(train_results[1])
        label_list = list(label_to_id.keys())
        cm = confusion_matrix(test_results[3], test_results[2], labels=list(label_to_id.values()))
        cm_display = ConfusionMatrixDisplay(cm, display_labels=label_list).plot()
        # cm = confusion_matrix(test_results[3], test_results[2])
        # cm_display = ConfusionMatrixDisplay(cm, display_labels=label_to_id.keys()).plot()
        plt.show()

        accuracy_dict[dict_name].append(accs)
        accuracy_dict[dict_name + "_final"].append(final_accs)
        accuracy_dict[dict_name + "_train"].append(train_accs)
        
        with open("../../data/probe_checkpoints/reading_probe_experiment.pkl", "wb") as outfile:
            pickle.dump(accuracy_dict, outfile)
    del dataset, train_dataset, test_dataset, train_loader, test_loader
    torch.cuda.empty_cache()

### Control Probe

In [ ]:
from probes import LinearProbeClassification, LinearProbeClassificationMixScaler
import sklearn.model_selection
import pickle
import random

jump_socioeco = True

new_prompt_format=True
residual_stream=True
uncertainty = False
logistic = True
augmented = False
remove_last_ai_response = True
include_inst = True
one_hot = True

label_to_id_age = {"child": 0,
                   "adolescent": 1,
                   "adult": 2,
                   "older adult": 3,
                  }

label_to_id_gender = {"male": 0,
                      "female": 1,
                     }

label_to_id_socioeconomic = {"low": 0,
                             "middle": 1,
                             "high": 2}

label_to_id_neweducation = {"someschool": 0,
                            "highschool": 1,
                            "collegemore": 2}
label_to_id_priority = {
    "platform": 0,
    "developer": 1,
    "user": 2
}

prompt_translator = {"_age_": "age",
                     "_gender_": "gender",
                     "_socioeco_": "socioeconomic status",
                     "_education_": "education level",
                     "_priority_": "priority level",
                    }

openai_dataset = {"_age_": "../../data/dataset/openai_age_1/",
                  "_gender_": "../../data/dataset/openai_gender_1/",
                  "_education_": "../../data/dataset/openai_education_1/",
                  "_socioeco_": "../../data/dataset/openai_socioeconomic_1/",
                  "_priority_": "../../data/dataset/openai_priority_academic_dishonesty",
                 }


accuracy_dict = {}

# directories = ["../../data/dataset/llama_age_1/", "../../data/dataset/llama_gender_1/",
#                "../../data/dataset/llama_socioeconomic_1/", "../../data/dataset/openai_education_1/"
#               ]
directories = ["../../data/dataset/openai_priority_academic_dishonesty/"]

# label_idfs = ["_age_", "_gender_", "_socioeco_", "_education_", "_priority_"]
label_idfs = ["_priority_"]

# label_to_ids = [label_to_id_age, label_to_id_gender,
#                 label_to_id_socioeconomic, label_to_id_neweducation,
#                 label_to_id_priority,
#                ]
label_to_ids = [label_to_id_priority]

for directory, label_idf, label_to_id in zip(directories, label_idfs, label_to_ids):
    # additional_dataset=[directory[:-1] + "_additional/"]
    if label_idf == "_education_":
        additional_dataset=[]
    elif label_idf == "_priority_":
        additional_dataset=[]
    else:
        additional_dataset=[directory[:-2] + "2/", openai_dataset[label_idf]]
    if label_idf == "_gender_":
        additional_dataset += ["../../data/dataset/openai_gender_2/", "../../data/dataset/openai_gender_3/", 
                               "../../data/dataset/openai_gender_4",]
    if label_idf == "_education_":
        additional_dataset += ["../../data/dataset/openai_education_2", "../../data/dataset/openai_education_3/"]
    if label_idf == "_socioeco_":
        additional_dataset += ["../../data/dataset/openai_socioeconomic_2/", "../../data/dataset/openai_socioeconomic_3/"]
    if label_idf == "_age_":
        additional_dataset += ["../../data/dataset/openai_age_2/"]
    if label_idf == "_priority_":
        additional_dataset += [
            "../../data/dataset/openai_priority_addiction_facilitation/",
            "../../data/dataset/openai_priority_animal_harm/", 
            "../../data/dataset/openai_priority_autonomous_agent_scope_creep/",
            "../../data/dataset/openai_priority_bioweapons/",
            "../../data/dataset/openai_priority_child_custody/",
            "../../data/dataset/openai_priority_child_safety/",
            "../../data/dataset/openai_priority_competitor_mentions/",
        ]
        
    dataset = TextDataset(directory, tokenizer, model, label_idf=label_idf, label_to_id=label_to_id,
                          convert_to_llama2_format=True, additional_datas=additional_dataset, 
                          new_format=new_prompt_format, control_probe=True,
                          residual_stream=residual_stream, if_augmented=augmented, 
                          remove_last_ai_response=remove_last_ai_response, include_inst=include_inst, k=1,
                          one_hot=False, last_tok_pos=-1)
    dict_name = label_idf.strip("_")

    train_size = int(0.8 * len(dataset))
    test_size = len(dataset) - train_size
    train_idx, val_idx = sklearn.model_selection.train_test_split(list(range(len(dataset))), 
                                                                  test_size=0.2,
                                                                  train_size=0.8,
                                                                  random_state=12345,
                                                                  shuffle=True,
                                                                  stratify=dataset.labels,
                                                                 )

    train_dataset = Subset(dataset, train_idx)
    test_dataset = Subset(dataset, val_idx)

    sampler = None
    train_loader = DataLoader(train_dataset, shuffle=True, sampler=sampler, pin_memory=True, batch_size=200, num_workers=1)
    test_loader = DataLoader(test_dataset, shuffle=False, pin_memory=True, batch_size=400, num_workers=1)

    if uncertainty:
        loss_func = edl_mse_loss
    else:
        loss_func = nn.BCELoss()
    torch_device = "cuda"

    seeds = seeds[:9]
    accuracy_dict[dict_name] = []
    accuracy_dict[dict_name + "_final"] = []
    accuracy_dict[dict_name + "_train"] = []
        
    accs = []
    final_accs = []
    train_accs = []
    for i in tqdm(range(0, 41)):
        trainer_config = TrainerConfig()
        probe = LinearProbeClassification(probe_class=len(label_to_id.keys()), device="cuda", input_dim=5120,
                                            logistic=logistic)
        optimizer, scheduler = probe.configure_optimizers(trainer_config)
        best_acc = 0
        max_epoch = 50
        verbosity = False
        layer_num = i
        print("-" * 40 + f"Layer {layer_num}" + "-" * 40)
        for epoch in range(1, max_epoch + 1):
            if epoch == max_epoch:
                verbosity = True
            # Get the train results from training of each epoch
            if uncertainty:
                train_results = train(probe, torch_device, train_loader, optimizer, 
                                        epoch, loss_func=loss_func, verbose_interval=None,
                                        verbose=verbosity, layer_num=layer_num, 
                                        return_raw_outputs=True, epoch_num=epoch, num_classes=len(label_to_id.keys()))
                test_results = test(probe, torch_device, test_loader, loss_func=loss_func, 
                                    return_raw_outputs=True, verbose=verbosity, layer_num=layer_num,
                                    scheduler=scheduler, epoch_num=epoch, num_classes=len(label_to_id.keys()))
            else:
                train_results = train(probe, torch_device, train_loader, optimizer, 
                                        epoch, loss_func=loss_func, verbose_interval=None,
                                        verbose=verbosity, layer_num=layer_num,
                                        return_raw_outputs=True,
                                        one_hot=one_hot, num_classes=len(label_to_id.keys()))
                test_results = test(probe, torch_device, test_loader, loss_func=loss_func, 
                                    return_raw_outputs=True, verbose=verbosity, layer_num=layer_num,
                                    scheduler=scheduler,
                                    one_hot=one_hot, num_classes=len(label_to_id.keys()))

            if test_results[1] > best_acc:
                best_acc = test_results[1]
                torch.save(probe.state_dict(), f"../../data/probe_checkpoints/controlling_probe/{dict_name}_probe_at_layer_{layer_num}.pth")
        torch.save(probe.state_dict(), f"../../data/probe_checkpoints/controlling_probe/{dict_name}_probe_at_layer_{layer_num}_final.pth")
        
        accs.append(best_acc)
        final_accs.append(test_results[1])
        train_accs.append(train_results[1])
        # cm = confusion_matrix(test_results[3], test_results[2])
        # cm_display = ConfusionMatrixDisplay(cm, display_labels=label_to_id.keys()).plot()
        label_list = list(label_to_id.keys())
        cm = confusion_matrix(test_results[3], test_results[2], labels=list(label_to_id.values()))
        cm_display = ConfusionMatrixDisplay(cm, display_labels=label_list).plot()
        plt.show()

        accuracy_dict[dict_name].append(accs)
        accuracy_dict[dict_name + "_final"].append(final_accs)
        accuracy_dict[dict_name + "_train"].append(train_accs)
        
        with open("../../data/probe_checkpoints/controlling_probe_experiment.pkl", "wb") as outfile:
            pickle.dump(accuracy_dict, outfile)
    del dataset, train_dataset, test_dataset, train_loader, test_loader
    torch.cuda.empty_cache()